# Mixed TTL Caching — Converse API

This notebook demonstrates mixed-TTL cache checkpoints — using different expiration times for different content sections in a single request — using the **Converse** and **ConverseStream** APIs.

## What you'll learn

- Assign different TTL values (e.g., 1 hour vs 5 minutes) to different cache checkpoints
- Read the `cacheDetails` response field for per-TTL token breakdowns
- TTL ordering constraint: longer TTL checkpoints must come before shorter ones

## Why mixed TTL?

- **Core reference material** (rarely changes) gets a longer TTL (e.g., `1h`)
- **Session-specific context** (changes more often) gets a shorter TTL (e.g., `5m`)

## Prerequisites

- AWS credentials configured (default profile)
- Access to Claude Sonnet 4.6 on Amazon Bedrock
- `boto3 >= 1.35.76`

## Token threshold

Claude Sonnet 4.6 requires at least **2,048 tokens** per cache checkpoint.

In [ ]:
!pip install --upgrade --quiet boto3

In [ ]:
import boto3
import json
import time

MODEL_ID = "global.anthropic.claude-sonnet-4-6"
AWS_REGION = "us-west-2"
CACHE_TTL_LONG = "1h"
CACHE_TTL_SHORT = "5m"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
print(f"Model: {MODEL_ID}")
print(f"TTLs: {CACHE_TTL_LONG} (section 1), {CACHE_TTL_SHORT} (section 2)")

## Sample Content

Two sections, each exceeding 2,048 tokens:
- **Section 1** (core reference): Inner + outer solar system overview — cached at 1h TTL
- **Section 2** (session context): Exoplanets + space exploration — cached at 5m TTL

In [ ]:
SPACE_SECTION_1 = """
The universe is a vast and mysterious expanse that has captivated human imagination for millennia. From the earliest civilizations who looked up at the night sky and wondered about the nature of the stars, to modern astronomers using sophisticated telescopes and spacecraft to explore distant galaxies, our quest to understand the cosmos continues unabated.

Our solar system, located in the Milky Way galaxy, is home to eight planets, numerous dwarf planets, and countless smaller objects including asteroids, comets, and meteoroids. The Sun, a middle-aged G-type main-sequence star, provides the energy that sustains life on Earth and influences the dynamics of all objects within its gravitational reach.

Mercury, the innermost planet, experiences extreme temperature variations due to its proximity to the Sun and lack of substantial atmosphere. Venus, often called Earth's twin due to its similar size, has a thick atmosphere composed primarily of carbon dioxide, creating a runaway greenhouse effect that makes it the hottest planet in our solar system. Earth, our home, is the only known planet to harbor life, with its unique combination of liquid water, moderate temperatures, and protective magnetic field.

Mars, the Red Planet, has long been a subject of fascination and speculation about the possibility of extraterrestrial life. Its rusty appearance comes from iron oxide prevalent on its surface. The planet features the largest volcano in the solar system, Olympus Mons, and a canyon system, Valles Marineris, that dwarfs the Grand Canyon. Recent Mars missions have discovered evidence of ancient river systems and the presence of water ice beneath the surface.

The asteroid belt, located between Mars and Jupiter, contains millions of rocky objects ranging from small boulders to the dwarf planet Ceres. These remnants from the early solar system provide valuable insights into planetary formation and the conditions that existed billions of years ago. Scientists study these asteroids to understand the building blocks of planets and the early solar system's chemical composition. Some asteroids contain valuable metals and minerals that may one day be mined for space-based industries.

The terrestrial planets share common characteristics: rocky compositions, relatively small sizes compared to gas giants, and solid surfaces. Mercury's heavily cratered surface resembles our Moon, preserving a record of impacts from the early solar system. Venus's dense atmosphere traps heat so effectively that its surface temperature exceeds that of Mercury, despite being farther from the Sun. Earth's plate tectonics continuously reshape its surface, while Mars shows evidence of past geological activity including ancient volcanoes and water-carved channels.

Jupiter, the largest planet, is a gas giant composed primarily of hydrogen and helium. Its Great Red Spot, a persistent anticyclonic storm, has been observed for over 400 years. Jupiter's intense magnetic field and numerous moons, including the four Galilean satellites discovered by Galileo Galilei in 1610, make it a miniature solar system in its own right. Europa, one of these moons, is believed to have a subsurface ocean that could potentially harbor life.

Saturn, famous for its spectacular ring system, is another gas giant with dozens of moons. Titan, its largest moon, has a thick atmosphere and liquid hydrocarbon lakes, making it one of the most intriguing bodies in the solar system for astrobiological research. The Cassini-Huygens mission provided unprecedented details about Saturn and its moons during its 13-year exploration.

Uranus and Neptune, the ice giants, reside in the outer reaches of our solar system. Uranus rotates on its side, likely due to a massive impact early in its history. Neptune, the windiest planet, features storms with wind speeds exceeding 2,000 kilometers per hour. Both planets have ring systems, though less prominent than Saturn's.

Beyond Neptune lies the Kuiper Belt, a region populated by icy bodies including the dwarf planet Pluto. The New Horizons mission's flyby of Pluto in 2015 revealed a geologically active world with nitrogen glaciers and a hazy atmosphere. Even further out is the Oort Cloud, a hypothetical spherical shell of icy objects that may extend halfway to the nearest star.

The outer solar system remains largely unexplored compared to the inner planets. Only Voyager 2 has visited both Uranus and Neptune, conducting brief flybys in the 1980s. Future missions are being planned to study these ice giants in more detail, potentially including orbiters and atmospheric probes. The moons of the outer planets present exciting targets for astrobiology, with Europa, Enceladus, and Titan all showing signs of environments that could support life.

Gas giants and ice giants differ fundamentally in composition. While Jupiter and Saturn are primarily hydrogen and helium, Uranus and Neptune contain significant amounts of water, ammonia, and methane ices. This distinction gives the ice giants their characteristic blue-green colors and different internal structures compared to their larger neighbors.
"""

SPACE_SECTION_2 = """
Exoplanet research has revolutionized our understanding of planetary systems. The Kepler space telescope discovered thousands of planets orbiting other stars, revealing that planets are common throughout our galaxy. Some of these exoplanets reside in the habitable zone of their stars, where liquid water could exist on the surface.

The search for extraterrestrial intelligence, known as SETI, uses radio telescopes to listen for signals from advanced civilizations. While no definitive signals have been detected, the Drake Equation provides a framework for estimating the number of communicating civilizations in our galaxy.

Black holes, regions of spacetime where gravity is so strong that nothing can escape, represent some of the most extreme objects in the universe. Stellar black holes form from the collapse of massive stars, while supermassive black holes, containing millions to billions of solar masses, reside at the centers of most galaxies including our own.

The Event Horizon Telescope collaboration captured the first image of a black hole's shadow in 2019, confirming predictions from Einstein's general theory of relativity. Gravitational wave detectors have also observed the mergers of black holes and neutron stars, opening a new window on the universe.

Dark matter and dark energy together comprise about 95% of the universe's total mass-energy content. Dark matter, which does not emit or absorb light, reveals its presence through gravitational effects on visible matter. Dark energy, even more mysterious, appears to be driving the accelerated expansion of the universe.

The cosmic microwave background radiation, discovered in 1965, provides a snapshot of the universe approximately 380,000 years after the Big Bang. Detailed measurements of this radiation have confirmed the Big Bang theory and revealed information about the early universe's composition and geometry.

Modern telescopes have detected exoplanets using multiple methods including transit photometry, radial velocity measurements, and direct imaging. Transit photometry measures the slight dimming of starlight when a planet passes in front of its host star. Radial velocity detects the wobble in a star's motion caused by orbiting planets. Direct imaging captures actual light from exoplanets, though this remains challenging due to the overwhelming brightness of host stars.

Stellar evolution describes how stars change over their lifetimes. Stars form in molecular clouds when gravity causes dense regions to collapse. Nuclear fusion in the core converts hydrogen to helium, releasing enormous amounts of energy. When stars exhaust their nuclear fuel, their fate depends on their mass: smaller stars become white dwarfs, medium stars may become neutron stars, and the most massive stars explode as supernovae, potentially leaving behind black holes.

Galaxies, containing billions of stars, come in various shapes including spiral, elliptical, and irregular. The Milky Way is a barred spiral galaxy approximately 100,000 light-years in diameter. Galaxies often cluster together, forming groups and superclusters connected by cosmic filaments of dark matter and gas.

Space exploration has achieved remarkable milestones since the launch of Sputnik in 1957. Human spaceflight began with Yuri Gagarin's orbit in 1961 and culminated in the Apollo Moon landings. The International Space Station has hosted continuous human presence in space since 2000. Future missions aim to return humans to the Moon and eventually send astronauts to Mars.

Private space companies have transformed the industry, developing reusable rockets that have dramatically reduced launch costs. These advances have enabled new possibilities for satellite constellations, space tourism, and lunar exploration. Plans for permanent lunar bases and Mars colonies represent the next frontier of human expansion into space.

The study of astrobiology examines the origin, evolution, and distribution of life in the universe. Scientists search for biosignatures in planetary atmospheres and analyze extremophiles on Earth to understand the limits of life. The discovery of organic molecules on Mars and in the plumes of Enceladus fuels speculation about the possibility of life elsewhere in our solar system.

Space telescopes have revolutionized astronomy by observing wavelengths blocked by Earth's atmosphere. The Hubble Space Telescope has provided stunning images and groundbreaking discoveries for over three decades. The James Webb Space Telescope, launched in 2021, observes in infrared to study the earliest galaxies and probe planetary atmospheres for signs of life.

Understanding the universe requires collaboration across disciplines including physics, chemistry, biology, and engineering. International cooperation has enabled ambitious projects like the International Space Station and large-scale observatories. The quest to explore space continues to inspire new generations of scientists and engineers who will push the boundaries of human knowledge and capability.
"""

QUESTION = "Based on the context above about space, what are the two ice giant planets in our solar system and what makes them unique?"

## 1. Converse API — Mixed TTL

Each `cachePoint` can have its own `ttl` value. Longer TTL checkpoints must come before shorter ones:

```python
content = [
    {"text": "<core reference>"},
    {"cachePoint": {"type": "default", "ttl": "1h"}},   # longer TTL first
    {"text": "<session context>"},
    {"cachePoint": {"type": "default", "ttl": "5m"}},   # shorter TTL second
    {"text": "<question>"}
]
```

In [ ]:
def converse_mixed_ttl():
    content = [
        {"text": SPACE_SECTION_1},
        {"cachePoint": {"type": "default", "ttl": CACHE_TTL_LONG}},
        {"text": SPACE_SECTION_2},
        {"cachePoint": {"type": "default", "ttl": CACHE_TTL_SHORT}},
        {"text": QUESTION}
    ]

    response = bedrock.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": 512}
    )
    return response["usage"]

# Request 1 — cache write
usage1 = converse_mixed_ttl()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_mixed_ttl()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## Understanding `cacheDetails`

The `cacheDetails` field in the Converse API response shows how many tokens are cached at each TTL. This helps you verify that both TTL tiers are working correctly.

Example:
```json
"cacheDetails": [
    {"ttl": "1h", "inputTokens": 1547},
    {"ttl": "5m", "inputTokens": 1823}
]
```

## 2. ConverseStream — Mixed TTL with Streaming

Same mixed TTL pattern with streaming output.

In [ ]:
def converse_stream_mixed_ttl():
    content = [
        {"text": SPACE_SECTION_1},
        {"cachePoint": {"type": "default", "ttl": CACHE_TTL_LONG}},
        {"text": SPACE_SECTION_2},
        {"cachePoint": {"type": "default", "ttl": CACHE_TTL_SHORT}},
        {"text": QUESTION}
    ]

    response = bedrock.converse_stream(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": 512}
    )

    text = ""
    usage = {}
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            text += event["contentBlockDelta"]["delta"].get("text", "")
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})

    return usage

# Request 1 — cache write
usage1 = converse_stream_mixed_ttl()
print("Request 1 (cache write expected):")
print(json.dumps(usage1, indent=2))

time.sleep(1)

# Request 2 — cache read
usage2 = converse_stream_mixed_ttl()
print("\nRequest 2 (cache read expected):")
print(json.dumps(usage2, indent=2))

## TTL Ordering Constraint

Cache checkpoints must be ordered from **longest to shortest TTL** within a single request. The API will return an error if a shorter TTL appears before a longer one.

Valid: `1h` then `5m`

Invalid: `5m` then `1h`

This constraint applies across all cache checkpoint locations (messages, system, tools).

## Conclusion

Mixed TTL caching allows fine-grained control over cache lifetimes:
- Use longer TTLs for stable reference content (domain knowledge, product catalogs)
- Use shorter TTLs for session-specific context that changes more frequently
- Monitor `cacheDetails` to verify per-TTL token distribution

The `cachePoint` syntax with `ttl` is model-agnostic — works with any Bedrock model that supports prompt caching.

For more details, see the [Amazon Bedrock Prompt Caching documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/prompt-caching.html).